# Final Project
Takahiro Hikida

Date - 16 May 2026

## Introduction
### Motivation
When the Federal Reserve (US central bank) raises interest rates or unemployment spikes, the impact on Economic Growth (Real GDP) doesn’t happen overnight. Instead, economic variables act like falling dominoes, where a shock in one area creates a ripple effect that takes months or even years to fully materialize.

While simple correlations can show us what the economy looks like at a specific snapshot in time, they fail to capture these crucial time delays (lags) and chained reactions. In this project, I attempted to create a mathematical model of economic growth, taking these time delays into account.

### Questions asked
1. How has the US economy evolved over time?
2. How long do economic variables take to affect the Economic Growth?
3. Does “Consumer Sentiment” (how people feel about the economy) actually drive real Economic Growth, or is it merely statistical noise?
4. To what extent can we mathematically model and predict the structural portion of Economic Growth, separating it from unpredictable external shocks?

### The Approach
To answer the questions above, I had to learn and implement a new technique: the VAR (Vector AutoRegression) model and Impulse Response Function (IRF) analysis.

Unlike simple linear regressions, a VAR model consists of a system of equations that allows multiple variables to influence each other over time. By combining this advanced time-series technique with interactive Plotly visualizations, I have built a data narrative that simulates how an initial “shock” (like a sudden rate hike) ripples through the US economy.

In [11]:
#Import necessary packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go

## Procedure
### 1. Data collection and wrangling
I used the data all from Federal Reserve Bank of St.Louis as csv files.

* University of Michigan: Consumer Sentiment (UMCSENT) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/UMCSENT)
* Federal Funds Effective Rate (FEDFUNDS) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/FEDFUNDS)
* Consumer Price Index for All Urban Consumers: All Items in U.S. City Average (CPIAUCSL) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/CPIAUCSL)
* Unemployment Rate (UNRATE) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/UNRATE)
* Real Gross Domestic Product (GDPC1) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/GDPC1)

In [12]:
#Call the data
from google.colab import drive
drive.mount('/content/drive')
df1=pd.read_csv("/content/drive/MyDrive/CS215-Spring-TakahiroHikida/Data/GDPC1.csv")
df2=pd.read_csv("/content/drive/MyDrive/CS215-Spring-TakahiroHikida/Data/CPIAUCSL.csv")
df3=pd.read_csv("/content/drive/MyDrive/CS215-Spring-TakahiroHikida/Data/UNRATE.csv")
df4=pd.read_csv("/content/drive/MyDrive/CS215-Spring-TakahiroHikida/Data/FEDFUNDS.csv")
df5=pd.read_csv("/content/drive/MyDrive/CS215-Spring-TakahiroHikida/Data/UMCSENT.csv")

df1.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,observation_date,GDPC1
0,1947-01-01,2182.681
1,1947-04-01,2176.892
2,1947-07-01,2172.432
3,1947-10-01,2206.452
4,1948-01-01,2239.682


We will now create a large dataframe on time. I used an inner join because the range of time the data was taken was different across variables.

In [13]:
#Merge all together
df=pd.merge(df1,df2,on="observation_date", how="inner")
df=pd.merge(df,df3,on="observation_date", how="inner")
df=pd.merge(df,df4,on="observation_date", how="inner")
df=pd.merge(df,df5,on="observation_date", how="inner")
df.head()

,observation_date,GDPC1,CPIAUCSL,UNRATE,FEDFUNDS,UMCSENT
0,1954-07-01,2880.482,26.840,6.0,1.03,NaN
1,1954-10-01,2936.852,26.757,5.3,0.99,87.0
2,1955-01-01,3020.746,26.793,4.7,1.34,95.9
3,1955-04-01,3069.910,26.757,4.4,1.50,99.1
4,1955-07-01,3111.379,26.777,4.1,1.94,NaN


We can see a couple of NaNs in UMCSENT. I need a complete sequence of rows without NaNs for my analysis, so the data will start from the row after the last NaN shows up.

In [14]:
#Clean up df, i.e., get a sequence of data without NaNs.
nan_rows=df[df.isna().any(axis=1)] #Subset the rows with NaN values
n=nan_rows.index.max() #Define the latest row with NaN
df=df.loc[df.index>n] #Cut every row above n off df
df

,observation_date,GDPC1,CPIAUCSL,UNRATE,FEDFUNDS,UMCSENT
21,1959-10-01,3439.832,29.370,5.6,3.99,93.8
22,1960-01-01,3517.181,29.397,5.1,3.93,100.0
23,1960-04-01,3498.246,29.573,5.2,3.70,93.3
24,1960-07-01,3515.385,29.590,5.5,2.94,97.2
25,1960-10-01,3470.278,29.780,6.3,2.30,90.1
...,...,...,...,...,...,...
282,2025-01-01,23548.210,319.475,4.1,4.33,64.5
283,2025-04-01,23770.976,320.786,4.2,4.33,55.0
284,2025-07-01,24026.834,323.235,4.3,4.29,58.3
285,2025-10-01,24055.749,325.547,4.5,3.90,52.5


Next, I will need to change the "observation_date" inputs into a datetime object. This is crucial because there will be a mess in the x-axis without doing so.

In [15]:
#Change the "observation_date" column into datetime object
df['observation_date']=pd.to_datetime(df['observation_date'])
#Rename columns
df=df.rename(columns={"observation_date":"Date","GDPC1":"Real GDP","CPIAUCSL":"CPI","UNRATE":"Unemployment Rate","FEDFUNDS":"Federal Funds Rate","UMCSENT":"Consumer Centiment"})

### 2. Visualizing the Macroeconomic Landscape over Time
Now, I would like to plot the variables in different graphs over time. I will try adding a dropdown button to the plot to make it more interactive.

I learned the procedure of doing this from the official Plotly site:

https://plotly.com/python/dropdowns/

In [16]:
#Plot every variable over time
fig=go.Figure() #Define the name of the plot

variables=df.columns[1:].tolist()  #Subset df excluding the "Date" column and make it a list

# Iterate through the list of variables to add a line for each one
for i, var in enumerate(variables): #enumerate(variables) allows us to extract not only the data but also the index number
    fig.add_trace(
        go.Scatter(
            x=df['Date'], # Set the x-axis to the observation dates
            y=df[var], # Set the y-axis to the current variable's values
            name=var, # Name the line after the variable
            visible=(i==0) # Make only the first variable visible by default, hide the rest
        )
    )

# Create an empty list to store the dropdown menu buttons
buttons = []

# Loop through the variables again to create a button for each one
for i, var in enumerate(variables):
    # Create a list of False values for all traces
    visibility = [False]*len(variables)
    # Set only the current variable's index to 'True' so only one line shows at a time
    visibility[i]=True

    # Define the properties and actions of the button
    button = dict(
        label=var, # The text displayed on the dropdown menu
        method="update", # Specify that clicking the button updates the graph
        args=[
            {"visible": visibility}, # Apply the visibility settings defined above
            {"title": f"{var} over time", "yaxis": {"title": var}} # Update the plot title and y-axis label
        ]
    )
    buttons.append(button) # Add the constructed button to the buttons list

# Apply the dropdown menu and overall styling to the figure layout
fig.update_layout(
    updatemenus=[
        dict(
            active=0, # Set the first button as the default active option
            buttons=buttons, # Attach the list of buttons created above
            x=0.0, # Position the dropdown menu horizontally
            xanchor="left",
            y=1.15, # Position the dropdown menu vertically (slightly above the graph)
            yanchor="top"
        )
    ],
    title=f"{variables[0]} over time", # Set the initial plot title
    xaxis_title="Date", # Set the x-axis label
    yaxis_title=variables[0], # Set the initial y-axis label
    template="plotly_white" # Use a clean, white background theme
)

fig.show()

fig.write_html("variables.html")

#### Observations
* We can see spikes/unusual movements when crises happen, such as the 2008 crisis and COVID.
* As we can see, when the unemployment rate rose, Real GDP dropped; there is likely an inverse relationship between them.

Before we move on to the analysis, I would like to modify Real GDP and CPI. These are rather cumulative statistics than growth. Since I want to model the economic growth, I will use the rate of change in Real GDP(d/dt(Real GDP)) and the inflation rate(d/dt(CPI)) instead of raw data. (1 unit of time is defined as a quarter of a year.)

This is very important because every other variable is in the form of (d/dt(x)), and we need to align with that. I got to understand this through this article.

https://otexts.com/fpp3/stationarity.html

In [17]:
df["Economic Growth Rate"]=df["Real GDP"].pct_change() #Get the rate of change by calling pct_change()
df["Inflation Rate"]=df["CPI"].pct_change()
df=df.dropna() #Since we modeled growth, the top row of both rates is NaN
df

,Date,Real GDP,CPI,Unemployment Rate,Federal Funds Rate,Consumer Centiment,Economic Growth Rate,Inflation Rate
22,1960-01-01,3517.181,29.397,5.1,3.93,100.0,0.022486,0.000919
23,1960-04-01,3498.246,29.573,5.2,3.70,93.3,-0.005384,0.005987
24,1960-07-01,3515.385,29.590,5.5,2.94,97.2,0.004899,0.000575
25,1960-10-01,3470.278,29.780,6.3,2.30,90.1,-0.012831,0.006421
26,1961-01-01,3493.703,29.840,6.8,2.00,91.6,0.006750,0.002015
...,...,...,...,...,...,...,...,...
282,2025-01-01,23548.210,319.475,4.1,4.33,64.5,-0.001625,0.009119
283,2025-04-01,23770.976,320.786,4.2,4.33,55.0,0.009460,0.004104
284,2025-07-01,24026.834,323.235,4.3,4.29,58.3,0.010763,0.007634
285,2025-10-01,24055.749,325.547,4.5,3.90,52.5,0.001203,0.007153


### 3. Impulse Response Analysis
My next goal is to mathematically model the Economic Growth rate in terms of other variables. Noticing that not all economic variables act instantly on Economic Growth, but there are some that have a lag to be effective. To embed that into my analysis, I would like to introduce the VAR(Vector AutoRegression) model. The code outputs a statistical summary of the VAR model, providing coefficients and p-values that capture the time-delayed relationships between our economic variables. We will analyze these p-values to filter out insignificant noise and construct a data-driven formula for Economic Growth, laying the groundwork for our Impulse Response Analysis.

I learned how to code through the statsmodels official site:

https://www.statsmodels.org/stable/vector_ar.html

In [18]:
from statsmodels.tsa.api import VAR
df_subset=df.loc[:, 'Unemployment Rate':] #Subset df from "Unemployment Rate" to the right

model=VAR(df_subset) #Toss the subset into VAR

lag_order_results=model.select_order(maxlags=8) #Test models up to 8 lags to evaluate and compare their statistical performance

print(lag_order_results.summary()) #Print out the results

#Set the optimal lag that had the best score in AIC.
optimal_lag=lag_order_results.aic
#Print out which lag it was
print(f"Optimal lag: {optimal_lag}")
#Train (fit) the VAR model using the calculated optimal lag
var_result=model.fit(optimal_lag)

print(var_result.summary())

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning:

An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.



 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0      -11.04      -10.97   1.600e-05      -11.02
1      -18.93     -18.52*   6.000e-09      -18.76
2      -19.11      -18.35   5.035e-09     -18.80*
3     -19.13*      -18.02  4.928e-09*      -18.68
4      -19.08      -17.63   5.197e-09      -18.49
5      -19.08      -17.29   5.176e-09      -18.36
6      -19.04      -16.90   5.436e-09      -18.18
7      -18.96      -16.47   5.901e-09      -17.96
8      -18.93      -16.10   6.075e-09      -17.79
-------------------------------------------------
Optimal lag: 3
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Sat, 16, May, 2026
Time:                     04:11:39
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -18.0646
Nobs:                 

The output above looks overwhelming, but here's what we have to look at:

* VAR Order Selection (Finding the Optimal Lag)

The first table runs a tournament among different lag periods (from 0 to 8 quarters) to see which one creates the most accurate model. We look at the AIC (Akaike Information Criterion), which scores the models while penalizing for over-complexity. The asterisk * marks the lowest (best) score.
* Result of Optimal Lag Calculation

The model determined that an Optimal Lag of 3 is best. This means it takes up to 3 quarters (9 months) for an economic shock to fully ripple through the system.

* P-Values in the "Economic Growth Rate" Equation

The most important column here is prob (p-value). In statistics, a p-value of less than 0.05 means the variable is statistically significant (it genuinely affects Economic Growth).
* Significant Drivers

 L1.Unemployment Rate (p=0.000) and L2. Federal Funds Rate (p=0.006) is highly significant. This proves statistically that job losses and interest rate hikes actively drive economic growth up or down.
* The "Noise."

If we look at consumer sentiment across all lags, L1, L2, and L3, their p-values are 0.889, 0.323, and 0.955, way above 0.05. This mathematically confirms our earlier hypothesis: Consumer Sentiment does not drive Economic Growth; it is merely statistical noise in this structural model.

* Correlation Matrix of Residuals

At the very bottom, we see how the errors of these variables relate to each other.

 There is a strong negative correlation (-0.745) between the Unemployment Rate and the Economic Growth Rate. This perfectly aligns with economic reality (Okun's Law): when unemployment unexpectedly spikes, economic growth sharply drops.

#### Plot the significance of each variable on economic growth
I'd like to visually model the reliability of Economic Growth on other variables. To do this, I will assume there was a shock in one variable by 1 standard deviation upwards, everything else unchanged. Setting that time as t=0, I will model the fluctuation of the Economic Growth Rate over time, up to 10 quarters.

This was by far the hardest code I implemented in this project. There was a lot of stuff I had to learn to make this. I learned the code from the official sites:

https://plotly.com/python/continuous-error-bars/ <br>

https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.irf.IRAnalysis.html


In [19]:
from plotly.subplots import make_subplots #This is the package we need here

# Set the responding variable (The variable we want to observe the impact on)
response_var = 'Economic Growth Rate'

# Create a list of impulse variables (The variables that will cause the "shock")
impulse_vars = [col for col in df_subset.columns if col != response_var]
periods = 10 # Set the time horizon for the model to 10 quarters

# Generate the Impulse Response Function (IRF) from the fitted VAR model
irf = var_result.irf(periods)

# Extract the orthogonalized IRF values (main predictions) and their standard errors
irf_values = irf.orth_irfs
stderr = irf.stderr(orth=True)

# Initialize a vertical subplot layout for each impulse variable
fig = make_subplots(rows=len(impulse_vars), cols=1,
                    subplot_titles=[f"Shock from {v}" for v in impulse_vars])

# Find the specific index number of our response variable to extract the correct data array
res_idx = df_subset.columns.tolist().index(response_var)

# Loop through each impulse variable to draw its corresponding IRF graph
for i, imp_var in enumerate(impulse_vars):
    imp_idx=df_subset.columns.tolist().index(imp_var)

    # Extract the main predicted response line for the current impulse variable
    y_values=irf_values[:periods+1, res_idx, imp_idx]

    # Calculate the 95% Confidence Interval (Margin of Error) using the 1.96 multiplier
    y_upper=y_values + 1.96 * stderr[:periods+1, res_idx, imp_idx]
    y_lower=y_values - 1.96 * stderr[:periods+1, res_idx, imp_idx]

    x=list(range(periods + 1))

    # Draw the Confidence Interval (The shaded grey/green area)
    fig.add_trace(go.Scatter(
        x=x + x[::-1], # Connect the x-axis forward and backward to create a closed polygon
        y=list(y_upper) + list(y_lower)[::-1], # Connect upper bounds forward and lower bounds backward
        fill='toself', # Fill the inside of the polygon
        fillcolor='rgba(0,100,80,0.2)', # Semi-transparent color for the shaded area
        line=dict(color='rgba(255,255,255,0)'), # Make the outline invisible
        hoverinfo="skip", # Prevent the tooltip from popping up on the shaded area
        showlegend=False
    ), row=i+1, col=1)

    # Draw the main predicted line (The solid colored line)
    fig.add_trace(go.Scatter(
        x=x, y=y_values,
        mode='lines+markers',
        name=imp_var,
        line=dict(color='royalblue', width=2)
    ), row=i+1, col=1)

    # Draw the horizontal Zero Line (The baseline for statistical significance)
    fig.add_shape(type="line", x0=0, y0=0, x1=periods, y1=0,
                  line=dict(color="black", width=1, dash="dash"),
                  row=i+1, col=1)

# Modify the overall layout and dimensions of the figure
fig.update_layout(
    height=300 * len(impulse_vars), # Automatically adjust height based on the number of plots
    title_text=f"What drives Economic Growth? (Impacts on {response_var})",
    showlegend=False,
    template="plotly_white"
)
fig.update_xaxes(title_text="Quarters after shock")


fig.show()
fig.write_html("irf_impacts_on_growth.html")

### Observation

#### How to read the chart
* The Dashed Zero Line: Represents "no impact" on Economic Growth.
* The Solid Blue Line: The main predicted change in Economic Growth over 10 quarters following a sudden shock (a 1-standard-deviation increase) in a specific variable.
* The Shaded Green Area: The 95% Confidence Interval.
* The Golden Rule: If the shaded area crosses or touches the zero line, the impact is statistically insignificant. It means we cannot confidently say the shock has any real effect.

#### Key findings

1. Unemployment Hurts Immediately

On the unemployment plot, he blue line immediately drops, and the entire shaded area stays below the zero line for the first few quarters. This visually proves that unexpected job losses significantly and negatively drag down economic growth.

2. Interest Rates Have a Delayed Drag

A shock in the Federal Funds Rate doesn't crash the economy immediately. However, after a few quarters, the shaded area dips below zero. This perfectly illustrates the lag on the effect of monetary policy: when the central bank raises interest rates, it takes time to cool down the economy.

3. Consumer Sentiment

Unlike what one might expect, from what we saw in the previous section, human feelings aren't just statistical noise. The green confidence interval rises clearly above the zero line for the first few quarters. This provides mathematical proof that an optimistic public actually drives real economic growth, likely because confident consumers spend more money, creating a positive ripple effect throughout the economy.

## 4. Plot the predicted model vs actual data
Finally, I would like to create a plot comparing the predicted values generated by our VAR model against the actual Economic Growth Rate.

Visualizing this fit allows us to see how well our structural model captures real-world economic cycles. Along with the visual comparison, the following code calculates two critical performance metrics:
* R-squared: Represents the percentage of the economy's movements that can be explained purely by our delayed structural variables. The closer this is to 1, the better the model is.

Formula: $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$

${SS_{res}}$ represents the total squared error of our model's predictions.

${SS_{tot}}$ represents the total natural variance of the actual economic data around its average.


* RMSE (Root Mean Squared Error): Shows the average error margin of our predictions. The closer this is to 0, the better the prediction is.

Formula: $RMSE = \sqrt{\frac{SS_{res}}{N}}$

We will also embed the final mathematical equation, representing the factors that dictate the US economy growth rate, into the chart, keeping only the variables we previously proved to be statistically significant.

In [20]:
# Subset the actual Economic Growth Rate from our dataset
actual=df_subset['Economic Growth Rate']

# Subset the predicted Economic Growth Rate generated by the VAR model
predicted=var_result.fittedvalues['Economic Growth Rate']

# Align the actual data with the predicted data.
# Since our optimal lag is 3, the model cannot predict the first 3 quarters.
actual_matched=actual[3:]

# Calculate the accuracy metrics
#The formula for this is
# Calculate the Sum of Squares of Residuals (Errors)
ss_res=((actual_matched-predicted) ** 2).sum()
# Calculate the Total Sum of Squares (Variance of the actual data)
ss_tot=((actual_matched-actual_matched.mean()) ** 2).sum()
# Calculate R-squared (How much variance is explained by the model)
r_squared=1-(ss_res / ss_tot)

# Calculate RMSE (Root Mean Squared Error) to find the average prediction error
# Since this is the average error, it is just the total squared errors divided by the number of data points (ss_res/n)
rmse = np.sqrt(ss_res / len(actual_matched))

#Build the figure
fig = go.Figure()

# Create a solid blue line for the Actual historical data
fig.add_trace(go.Scatter(
    x=df_subset.index,
    y=actual,
    name='Actual',
    line=dict(color='blue', width=2) #Set the color and width of the line
))

# Create a dashed red line for the Predicted data from our model
# We start plotting from index 3 to match the lag offset
fig.add_trace(go.Scatter(
    x=df_subset.index[3:],
    y=predicted,
    name='Predicted',
    line=dict(color='red', width=2, dash='dash')
))

#Create the Text Box for the Equation & Metrics
# Format the final derived equation using HTML tags for subscripts and line breaks
#This code, I had Gemini help me with.
equation_text = (
    "<b>Derived Equation (Significant Variables Only):</b><br><br>"
    "Growth<sub>t</sub> = 0.33 * Growth<sub>t-1</sub> + 0.24 * Growth<sub>t-2</sub><br>"
    "&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;+ 0.008 * Unemp<sub>t-1</sub> - 0.005 * Unemp<sub>t-2</sub><br>"
    "&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- 0.003 * FedRate<sub>t-2</sub><br><br>"
    "<b>Model Accuracy:</b><br>"
    f"R-squared: {r_squared:.3f}<br>"
    f"RMSE: {rmse:.3f}"
)

# Embed the text box into the plot
fig.add_annotation(
    x=1.02, y=1.0, # Position the box just outside the right edge of the plot
    xref="paper", yref="paper", # Use relative coordinates for positioning
    xanchor="left",
    yanchor="top",
    text=equation_text,
    showarrow=False, # We don't need an arrow pointing to the graph
    font=dict(family="Courier New, monospace", size=12, color="black"),
    align="left",
    bgcolor="rgba(255, 255, 255, 0.9)", # Set a slightly transparent white background
    bordercolor="black",
    borderwidth=1,
    borderpad=10 # Add some padding inside the text box
)

# Finalize the Layout of the plot
fig.update_layout(
    title="Actual vs. Predicted Economic Growth Rate",
    xaxis_title="Time",
    yaxis_title="Economic Growth Rate",
    template="plotly_white",
    margin=dict(r=350) # Expand the right margin to make room for the equation text box
)

fig.write_html("equation_fit.html")
fig.show()

#### Observation

The graph above visualizes how well our mathematically derived equation performs against the actual historical data of the U.S. economy.

1. The Equation

By filtering out the statistical noise and keeping only the highly significant variables, we built a structural equation. It reveals that today's Economic Growth is fundamentally driven by:
* The growth rates from the past 1 to 2 quarters.
* The unemployment rate from the past 1 to 2 quarters.
* The interest rate (Federal Funds Rate) set by the Fed exactly 2 quarters ago.

It might seem wierd that Consumer Sentiment is missing from the derived equation above, even though our earlier IRF graphs proved it has a significant impact. This highlights the true power of the VAR model! The equation only shows direct impacts on GDP. Consumer Sentiment doesn't directly increase GDP; instead, it triggers a ripple effect (e.g., confident consumers spend more, which lowers unemployment), and those other variables then boost GDP two quarters later. The equation shows the last variable that causally affects economic growth, but the IRF simulation proves that Consumer Sentiment does indirectly affect economic growth.

In short, consumer sentiment was excluded because of its high p-values.

2. Interpreting accuracy
* An $R^2$ of 0.253 <br>
A 25% explanatory power might seem low. However, in macroeconomics, this is a profound finding. It mathematically proves that about 25% of the U.S. economy growth is strictly driven by the structural, delayed cycles of past unemployment, interest rates, and inflation.
* The Remaining 75% <br>
The moments where the actual data (Blue) drastically breaks away from our prediction (Red)—such as the massive 2020 spike—represent the remaining 75%. These are the unpredictable events, like global pandemics or sudden geopolitical crises, which cannot be forecasted merely by looking at last quarter's interest rates.

## Conclusion
This analysis yielded some insights into how the US economy actually operates beneath the surface. By moving beyond simple snapshots and building a time-delayed mathematical model, we answered our three (or four) core questions:

### 1. How has the US economy evolved over time?
Through our initial time-series analysis, we observed that the U.S. economy is highly cyclical but constantly disrupted by unpredictable external shocks, most notably the 2008 Financial Crisis and the 2020 COVID-19 pandemic. We also visually confirmed an inherent friction between indicators; for example, unemployment and economic growth consistently mirror each other in an inverse relationship.
### 2. How long do economic variables take to affect Economic Growth?
Our mathematical model proved that economic shocks do not happen overnight. The Vector AutoRegression (VAR) analysis revealed that variables like the Federal Funds Rate and Unemployment Rate act as strong, delayed forces. It takes several quarters for a hike in interest rates or a spike in job losses to fully seep into the system and drag down the overall Economic Growth Rate.
### 3. Does “Consumer Sentiment” actually drive real Economic Growth, or is it merely statistical noise?
It is absolutely **NOT** just noise! If we only looked at direct mathematical equations, we might mistakenly conclude that human emotions don't matter. However, our VAR simulation (IRF) proved that an optimistic public creates a powerful domino effect. An increase in Consumer Sentiment indirectly drives real Economic Growth exactly two quarters (6 months) later. This mathematically proves that how people feel about the economy eventually translates into real-world economic activity.
### 4. To what extent can we mathematically model and predict the structural portion of Economic Growth?
By stripping away the statistical noise and focusing only on structurally significant variables with optimal lags, we successfully built a predictive equation. Our model achieved an $R^2$ score of 0.253. This mathematically proves that about 25.3% of America's Economic Growth is strictly determined by the delayed cycles of past economic indicators. The remaining 75% represents the unpredictable events (like global pandemics) that cannot be forecasted by structural models.

## Reflection
### Challenges & Learning a New Technique
The most challenging part of this project was the implementation of the Vector AutoRegression (VAR) model and Impulse Response Functions (IRF). Understanding why we need to convert raw GDP into Growth Rates was a steep learning curve. Furthermore, interpreting a system of equations where variables impact each other with varying time lags required a completely different way of thinking compared to standard linear regression. However, this challenge was very interesting, that I now feel that I have discovered completely different from what I have been learning before.

###What surprised me the most
The biggest surprise was the difference between direct mathematical equations and VAR simulations (IRF). I initially thought Consumer Sentiment was just statistical noise because its p-values were not significant in the direct equation for GDP. However, by looking closely at the IRF graphs, I discovered it actually has a significant indirect impact on the economy exactly two quarters later. This realization taught me that the interpretation can be different across analyses, and we cannot fully rely on one of them, as we will find ourselves missing some intuition otherwise.

### Future Extensions
If I had more time and resources, I would extend this project by introducing. variables that capture global supply chain metrics or energy prices (like crude. oil). I would also be interested in applying this exact same VAR framework to other countries, such as Japan or the Eurozone, to see if their economies remember shocks for longer or shorter periods than the U.S.

## References & Resources
**Data Sources**
* University of Michigan: Consumer Sentiment (UMCSENT) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/UMCSENT)
* Federal Funds Effective Rate (FEDFUNDS) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/FEDFUNDS)
* Consumer Price Index for All Urban Consumers: All Items in U.S. City Average (CPIAUCSL) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/CPIAUCSL)
* Unemployment Rate (UNRATE) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/UNRATE)
* Real Gross Domestic Product (GDPC1) from Federal Reserve Bank of St.Louis (https://fred.stlouisfed.org/series/GDPC1)
* Plotly Dropdown Button: https://plotly.com/python/dropdowns/
* Stationarity of Data: https://otexts.com/fpp3/stationarity.html
* VAR Model Implementation: [Official statsmodels VAR documentation](https://www.statsmodels.org/stable/vector_ar.html)
* IRF(Impulse Response Function): https://www.statsmodels.org/stable/generated/statsmodels.tsa.vector_ar.irf.IRAnalysis.html
* Plotting Error Bands: [Plotly Continuous Error Bands](https://plotly.com/python/continuous-error-bars/)
* **AI Assistance:** Gemini (for code debugging, optimization)